### Notebook to build composite video around beat and subdivision

In [ ]:
import os
import pickle
import subprocess
# import pandas as pd
from utils_mocap_viz.generate_views import (    # organize this
    # get_output_dir,
    prepare_videos
)

from utils_mocap_viz.animated_merged_phase_analysis import animate_merged_phase_analysis
from utils_dance_anim.dance_dot import animate_dance_phase_analysis
from utils_composite_video.by_time_segment import process_layouts, concatenate_and_overlay_videos

from utils_pipeline.pipeline_B import *

### Config

In [19]:
file_name = "BKO_E1_D1_02_Maraka"
traj_dir  = "traj_files_presentation"
status    = "included"   # or "excluded"
traj_threshold = "0.001"        # or any other threshold

bvh_dir = os.path.join("data", "bvh_files")
bvh_file = os.path.join(bvh_dir, file_name + "_T")

# path to onsets and cycles csv files
cycles_csv_path = f"data/virtual_cycles/{file_name}_C.csv"
onsets_csv_path = f"data/drum_onsets/{file_name}.csv"
dance_csv_path = f"data/dance_onsets/{file_name}_T_dance_onsets.csv"

cycle_segs, windows = compute_windows(traj_dir, file_name, status, traj_threshold)

Loaded cycle_segments from BKO_E1_D1_02_Maraka_included_0.001.pkl
cycle_segments: [(166.55722222199998, 168.02211111066666), (168.02211111066666, 169.503), (169.503, 170.9785555553333), (170.9785555553333, 172.4505555553333), (172.4505555553333, 173.9083333333333), (173.9083333333333, 175.364333333), (175.364333333, 176.802555555), (176.802555555, 178.222111111), (178.222111111, 179.70655555533332), (179.70655555533332, 181.18477777766668), (181.18477777766668, 182.6696666665), (182.6696666665, 184.1696666663333), (184.1696666663333, 185.65322222166665), (185.65322222166665, 187.141222222), (187.141222222, 188.6505555553333), (188.6505555553333, 190.09233333300003), (190.09233333300003, 191.51455555533332), (191.51455555533332, 192.92166666633327), (192.92166666633327, 194.34122222200003), (194.34122222200003, 195.73766666633333), (195.73766666633333, 197.180333333), (197.180333333, 198.59099999966668), (198.59099999966668, 199.94477777766667), (199.94477777766667, 201.261222222), (201

### Generate trajectory video + trimmed dance video plots

In [11]:
# 'beat_1', 'beat_2', 'beat_3', 'beat_4', 
# 'subdiv_2', 'subdiv_3', 'subdiv_5', 'subdiv_6', 
# 'subdiv_8', 'subdiv_9', 'subdiv_11', 'subdiv_12'

w_key = "beat_1"
traj_tuples = windows[w_key][:2]
# traj_tuples = random.sample(windows[w_key], 2)  # Randomly sample 2 tuples from the list
traj_tuples

[(165.0923333333333, 168.02211111066666, 166.55722222199998),
 (166.54122222133333, 169.503, 168.02211111066666)]

In [ ]:
# to plot window around a beat, with 1 cycles before and after the beat,
# beat is placed at zero, and cycles are plotted around it.
# x axis ranges from -4 to 4, representing beat numbers

extract_centered_cycle_videos_and_plots(
    file_name = file_name,
    windows = traj_tuples,  # List of (win_start, win_end, t_poi) tuples, t_poi is the time of the beat/subbdiv placed at zero
    window_key = w_key,
    base_path_logs = "data/logs_v4_0.007_foot_jun3",            # logs_v4_0.007_foot_jun3       logs_v2_may
    figsize = (10, 3),
    dpi = 200,
    save_dir = "composite_videos",
    )

### Generate Skeletal video + trimmed_video_mix

In [ ]:
video_path = os.path.join("data", "videos", f"{file_name}_pre_R_Mix.mp4")

output_dir1 = os.path.join("composite_videos", file_name, w_key, "video_skeleton")
os.makedirs(output_dir1, exist_ok=True)

output_dir3 = os.path.join("composite_videos", file_name, w_key, "drum_dot_merged")
os.makedirs(output_dir3, exist_ok=True)

output_dir4 = os.path.join("composite_videos", file_name, w_key, "dance_dot")
os.makedirs(output_dir4, exist_ok=True)

views_to_generate = ['front']       # skeleton views ['front', 'right' 'left', 'top'] 

for start_time, end_time, _ in traj_tuples:

    save_fname = f"drum_dot_merged_{start_time:.2f}_{end_time:.2f}.mp4"
    animate_merged_phase_analysis(
        file_name, start_time, end_time,
        cycles_csv_path, onsets_csv_path,
        figsize=(10, 3), dpi=200,
        save_fname = save_fname,
        save_dir=output_dir3
        )
    
    animate_dance_phase_analysis(
        file_name, start_time, end_time,
        cycles_csv_path, dance_csv_path,
        figsize= (10, 3), dpi= 200, save_dir= output_dir4
    )

    view_videos = prepare_videos(
        filename= bvh_file,
        start_time= start_time,
        end_time= end_time,
        views_to_generate = views_to_generate,
        video_path= None,             # video_path, wonr generate video
        video_size= (1280, 720),
        fps= 24,
        output_dir = output_dir1
    )


### Generate animated kinematic plots

In [ ]:
def extract_cycle_plots(
    file_name: str,
    windows: list,  # List of (win_start, win_end, t_poi) tuples
    window_key: str,
    joint_name: str,
    axis: str = 'y',
    base_path_logs: str = "data/logs_v2_may",
    frame_rate: float = 240,  # Trajectory data frame rate
    n_beats_per_cycle: int = 4,
    n_subdiv_per_beat: int = 3,
    nn: int = 3,
    output_dir2: str = None,
    figsize: tuple = (10, 3),
    dpi: int = 200
):
    """
    Create trajectory animations for windows around points of interest (beats or subdivisions).
    Each plot shows [-cycle, 0-cycle, +cycle] around the POI.
    """
    bvh_to_mvnx = {
    'x': 'y',  # BVH side → MVNX side
    'y': 'z',  # BVH vertical → MVNX vertical
    'z': 'x',  # BVH forward → MVNX forward
    }
    
    # Load joint position data
    dir_csv = "extracted_mocap_csv"
    base_name = os.path.splitext(os.path.basename(file_name))[0]
    worldpos_file = os.path.join(dir_csv, f"{base_name}_T_worldpos.csv")
    
    try:
        world_positions = pd.read_csv(worldpos_file)
        print(f"Successfully loaded CSV with {len(world_positions)} rows")
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        raise
    
    # Get time column and position data
    time_column = world_positions.columns[0]  # First column is time
    times = world_positions[time_column].values
    positions = world_positions[f"{joint_name}.{axis.upper()}"].values
    
    print(f"\nProcessing {len(windows)} windows")
    # print(f"Total frames in trajectory data: {len(times)}")
    # print(f"Time range in trajectory data: {times[0]:.3f} to {times[-1]:.3f}")
    
    # Process each window
    for i, (win_start, win_end, t_poi) in enumerate(windows):
        print(f"\nProcessing window {i+1}:")
        print(f"  Window time range: {win_start:.3f} to {win_end:.3f}")
        
        # Calculate segment times
        start_time = win_start
        end_time = win_end
        duration = end_time - start_time
        downbeat = t_poi  # This is the point of interest (beat or subdivision)
        
        # Calculate avg_cycle from the window duration
        avg_cycle = duration / 2  # Since window is ±1 cycle
        
        # Calculate window parameters
        beat_len = avg_cycle / n_beats_per_cycle
        subdiv_len = beat_len / n_subdiv_per_beat
        half_win = subdiv_len * nn
        
        # Calculate frame numbers for trajectory (240fps)
        traj_start_frame = int(start_time * frame_rate)
        traj_end_frame = int(end_time * frame_rate)
        traj_n_frames = traj_end_frame - traj_start_frame
        
        print(f"  Trajectory frames: {traj_start_frame} to {traj_end_frame} (240fps)")
        
        # Check if we have valid frame numbers
        if traj_start_frame >= traj_end_frame:
            print(f"  Skipping window {i+1}: Invalid frame range (start >= end)")
            continue
        if traj_start_frame < 0:
            print(f"  Skipping window {i+1}: Start frame < 0")
            continue
        if traj_end_frame > len(positions):
            print(f"  Skipping window {i+1}: End frame > total frames")
            continue
        
        # Trim trajectory data using frame numbers at 240fps
        pos_win = positions[traj_start_frame:traj_end_frame]
        t_win = times[traj_start_frame:traj_end_frame]
        
        # Check if we have valid trajectory data
        if len(pos_win) == 0:
            print(f"  Skipping window {i+1}: No trajectory data")
            continue
        
        print(f"  Trajectory data points: {len(pos_win)}")
        
        # Create figure and axis
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        fig.tight_layout(pad=2.0) 
        
        # Calculate all subdivision times for the window
        all_subdiv_times = []
        for beat_idx in range(-n_beats_per_cycle, n_beats_per_cycle + 1):
            beat_time = downbeat + beat_idx * beat_len
            for subdiv_idx in range(n_subdiv_per_beat):
                subdiv_time = beat_time + subdiv_idx * subdiv_len
                if start_time <= subdiv_time <= end_time:
                    all_subdiv_times.append((subdiv_time, beat_idx * n_subdiv_per_beat + subdiv_idx + 1))

        # Plot subdivision lines with appropriate colors
        for subdiv_time, subdiv_num in all_subdiv_times:
            color = get_subdiv_color(subdiv_num)
            # Add yellow glow effect for t=0
            if abs(subdiv_time - downbeat) < 0.001:  # If it's the POI
                ax.axvline(subdiv_time, color='yellow', linestyle='-', linewidth=3, alpha=0.3)
            # ax.axvline(subdiv_time, color=color, linestyle='-', linewidth=2, alpha=0.7)
            
            if subdiv_num in [-11, -8, -5, -2, 1, 4, 7, 10]:
                ax.axvline(subdiv_time, color=color, linestyle='-', linewidth=2, alpha=0.7) #beat color
            else:
                ax.axvline(subdiv_time, color=color, linestyle='--', linewidth=1, alpha=0.3) #subdivision color
        
        # Plot trajectory
        ax.plot(t_win, pos_win, '-', color='green', alpha=0.5, label=f'{joint_name} {axis.upper()}')
        
        ax.axvline(downbeat, color='yellow', linewidth=6, alpha=0.3, zorder=0)  # Glow effect
        
        # Set y-axis limits with safety checks
        try:
            y_min = pos_win.min()
            y_max = pos_win.max()
            y_range = y_max - y_min
            ax.set_ylim(y_min - 0.1*y_range, y_max + 0.1*y_range)
        except ValueError as e:
            print(f"  Warning: Could not set y-axis limits: {e}")
            # Set default y-axis limits
            ax.set_ylim(-1, 1)
        
        # Create vertical playhead
        v_playhead, = ax.plot([start_time, start_time], 
                            [y_min - 0.1*y_range, y_max + 0.1*y_range],
                            lw=1, alpha=0.9, color='orange')
        
        # Set up the plot with scaled x-axis
        ax.set_xlabel(f'Beats relative to {window_key}')
        ax.set_ylabel(f'{joint_name} {bvh_to_mvnx[axis.lower()]} Position')      # {axis.upper()} y is vertical in bvh files, Z is vertical in mocap
        ax.set_title(f'Window {i+1} | {window_key}: {downbeat:.2f}s')
        ax.grid(True, alpha=0.3)
        
        # Scale x-axis to show beats instead of cycles
        x_ticks = np.arange(-n_beats_per_cycle, n_beats_per_cycle + 1)
        x_tick_positions = downbeat + x_ticks * beat_len
        ax.set_xticks(x_tick_positions)
        ax.set_xticklabels(x_ticks)
        
        # Add legend
        custom = [
            Line2D([0],[0], color='blue', lw=1),
            Line2D([0],[0], color='black', lw=1),
            Line2D([0],[0], color='green', lw=1),
            Line2D([0],[0], color='red', lw=1),
        ]
        labels = [
            f"{joint_name} {axis.upper()}", 
            "Subdiv-1 (1,4,7,10)", 
            "Subdiv-2 (2,5,8,11)", 
            "Subdiv-3 (3,6,9,12)"
        ]
        ax.legend(custom, labels, loc='upper left', framealpha=0.3, fontsize=6)
        
        def update(frame):
            v_playhead.set_xdata([frame, frame])
            ax.set_title(f'Cycle {i+1} | {window_key}: {downbeat:.2f}s | Time: {frame:.2f}s')
            return v_playhead,
        
        # Create animation frames at 24fps
        n_frames = int(duration * 24)
        frames = np.linspace(start_time, end_time, n_frames)
        anim = animation.FuncAnimation(
            fig, update, frames=frames,
            interval=1000/24,  # 24fps
            blit=True
        )
        
        # Save animation
        plot_output_path = os.path.join(output_dir2, f"{file_name}_window_{i+1:03d}_{start_time:.2f}_{end_time:.2f}.mp4")
        writer = animation.FFMpegWriter(fps= 24, 
                                        bitrate=2000,
                                        codec='libx264',  # Specify codec
                                        # extra_args=['-preset', 'ultrafast']
                                        )  # 24fps
        anim.save(plot_output_path, writer=writer)
        plt.close(fig)
        
        print(f"  Plot saved: {plot_output_path}")
        print(f"  Plot duration: {len(frames)/24:.3f}s")
    
    print("\nProcessing complete!")

In [ ]:
# Available markers: ['Hips', 'LeftHip', 'LeftKnee', 'LeftAnkle', 'LeftToe', 
# 'LeftToeEnd', 'RightHip', 'RightKnee', 'RightAnkle', 'RightToe', 'RightToeEnd', 
# 'Chest', 'Chest2', 'Chest3', 'Chest4', 'LeftCollar', 'LeftShoulder', 'LeftElbow', 
# 'LeftWrist', 'LeftWristEnd', 'RightCollar', 'RightShoulder', 'RightElbow', 
# 'RightWrist', 'RightWristEnd', 'Neck', 'Head', 'HeadEnd']
mvnx_to_bvh = {
    'x': 'z',  # forward mvnx → backward bvh
    'y': 'x',  # side mvnx → side bvh
    'z': 'y',  # vertical mvnx → vertical bvh
}


joint_name = "Hips"  
axis = 'z'      # z is vertical in mvnx files

output_dir2 = os.path.join("composite_videos", file_name, w_key, joint_name)
os.makedirs(output_dir2, exist_ok=True)

extract_cycle_plots(
file_name= file_name,
windows= traj_tuples,
window_key= w_key,
joint_name= joint_name,
axis= mvnx_to_bvh[axis],
output_dir2= output_dir2,
figsize = (10, 3),  # 2000 x 600 px
dpi= 200,
)

### Prepare for concatenation

In [18]:
vid_plot_path = os.path.join("composite_videos", file_name, w_key)

concatenate_and_overlay_videos(file_name, joint_name, vid_plot_path, views_to_generate)        # modify

concat_file_list = [f for f in os.listdir(vid_plot_path) if f.lower().endswith(".mp4")]
concat_dict = {
    f.replace('_concat.mp4', ''): os.path.join(vid_plot_path, f) 
    for f in concat_file_list
}

view_videos = {
    'video_mix': concat_dict['video_mix'],  
    'plot': concat_dict['plot'],    
    'joint_Hips': concat_dict['joint_Hips'],
    'drum_dot': concat_dict['drum_dot'],
    'dance_dot': concat_dict['dance_dot'],
    
    **{f'{view}_view': concat_dict[f'{view}_view'] for view in views_to_generate}
}

Concatenation files already exist, skipping creation
Concatenation complete: composite_videos/BKO_E1_D1_02_Maraka/beat_1


### Generate Composite Video

In [ ]:
view_videos = {
    'video_mix': concat_dict['video_mix'],  
    'plot': concat_dict['plot'],    
    'joint_Hips': concat_dict['joint_Hips'],
    'drum_dot': concat_dict['drum_dot'],
    'dance_dot': concat_dict['dance_dot'],
    
    'front_view': concat_dict['front_view'],
    # 'left_view': concat_dict['left_view'],
    # 'right_view': concat_dict['right_view'],
    # 'top_view': concat_dict['top_view'],
}

canvas_w, canvas_h = 1920, 1080 
composite_layout_1 = [
    # Top row - side by side
    {'view': 'video_mix', 'x': 0, 'y': 0, 'width': 960, 'height': 540},
    {'view': 'front_view', 'x': 960, 'y': 0, 'width': 960, 'height': 540},
    
    # Bottom row - stacked vertically
    {'view': 'joint_Hips', 'x': 0, 'y': 540, 'width': 1920, 'height': 270},
    {'view': 'plot', 'x': 0, 'y': 810, 'width': 1920, 'height': 270},
]

composite_layout_2 = [
    # Top row - side by side
    {'view': 'video_mix', 'x': 0, 'y': 0, 'width': 960, 'height': 540},
    {'view': 'front_view', 'x': 960, 'y': 0, 'width': 960, 'height': 540},
    
    # Bottom row - stacked vertically
    {'view': 'joint_Hips', 'x': 0, 'y': 540, 'width': 960, 'height': 270},
    {'view': 'plot', 'x': 0, 'y': 810, 'width': 960, 'height': 270},
    
    # {'view': 'drum_dot', 'x': 960, 'y': 540, 'width': 960, 'height': 540},
    {'view': 'drum_dot', 'x': 960, 'y': 540, 'width': 960, 'height': 270},
    {'view': 'dance_dot', 'x': 960, 'y': 810, 'width': 960, 'height': 270},
]

layouts_to_export = {
                     "layout_5views": composite_layout_2
                     }

In [16]:

process_layouts(layouts_to_export, view_videos, vid_plot_path, file_name, "None", "None", "None")

Resizing succeeded, output saved to: composite_videos/BKO_E1_D1_02_Maraka/beat_1/temp_resized/video_mix_concat.mp4
Resizing succeeded, output saved to: composite_videos/BKO_E1_D1_02_Maraka/beat_1/temp_resized/front_view_concat.mp4
Resizing succeeded, output saved to: composite_videos/BKO_E1_D1_02_Maraka/beat_1/temp_resized/joint_Hips_concat.mp4
Resizing succeeded, output saved to: composite_videos/BKO_E1_D1_02_Maraka/beat_1/temp_resized/plot_concat.mp4
Resizing succeeded, output saved to: composite_videos/BKO_E1_D1_02_Maraka/beat_1/temp_resized/drum_dot_concat.mp4
Resizing succeeded, output saved to: composite_videos/BKO_E1_D1_02_Maraka/beat_1/temp_resized/dance_dot_concat.mp4


ffmpeg version N-121749-g203c6a93d7 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 8 (GCC)
  configuration: --prefix=/itf-fi-ml/home/sagardu/ffmpeg_build --pkg-config-flags=--static --extra-cflags=-I/itf-fi-ml/home/sagardu/ffmpeg_build/include --extra-ldflags=-L/itf-fi-ml/home/sagardu/ffmpeg_build/lib --extra-libs='-lpthread -lm' --bindir=/itf-fi-ml/home/sagardu/bin --enable-gpl --enable-libx264 --enable-shared
  libavutil      60. 17.100 / 60. 17.100
  libavcodec     62. 19.100 / 62. 19.100
  libavformat    62.  6.101 / 62.  6.101
  libavdevice    62.  2.100 / 62.  2.100
  libavfilter    11.  9.100 / 11.  9.100
  libswscale      9.  3.100 /  9.  3.100
  libswresample   6.  2.100 /  6.  2.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'composite_videos/BKO_E1_D1_02_Maraka/beat_1/temp_resized/video_mix_concat.mp4':
  Metadata:
    major_brand     : isom
    minor_version   : 512
    compatible_brands: isomiso2avc1mp41
    encoder         : Lavf62.6.101
  Duration: 00:00:06.

Video successfully created as composite_videos/BKO_E1_D1_02_Maraka/BKO_E1_D1_02_Maraka_None_layout_5views_None_None.mp4


[out#0/mp4 @ 0x1142280] video:4142KiB audio:144KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.102003%
frame=  144 fps=0.0 q=-1.0 Lsize=    4291KiB time=00:00:05.94 bitrate=5912.3kbits/s dup=0 drop=141 speed=7.72x elapsed=0:00:00.76    
[libx264 @ 0xf1d400] frame I:1     Avg QP:20.00  size:209654
[libx264 @ 0xf1d400] frame P:143   Avg QP:19.86  size: 28191
[libx264 @ 0xf1d400] mb I  I16..4: 100.0%  0.0%  0.0%
[libx264 @ 0xf1d400] mb P  I16..4:  3.5%  0.0%  0.0%  P16..4: 13.3%  0.0%  0.0%  0.0%  0.0%    skip:83.3%
[libx264 @ 0xf1d400] coded y,uvDC,uvAC intra: 50.0% 45.5% 30.0% inter: 7.0% 4.6% 1.2%
[libx264 @ 0xf1d400] i16 v,h,dc,p: 41% 48%  5%  6%
[libx264 @ 0xf1d400] i8c dc,h,v,p: 33% 40% 24%  3%
[libx264 @ 0xf1d400] kb/s:5654.64
[aac @ 0x1160c40] Qavg: 672.401
